# ATML PA2 - Task 1: DPO

**Before running:** Settings → Accelerator **GPU T4** (x1 or x2), Internet **On**.

Already run (results committed). This notebook reproduces it. Rough T4 time: ~3-3.5 h.
`outputs/task1_dpo/standard` is the DPO policy needed in Task 4 - keep the zip.

## 1. Setup

In [ ]:
# Fresh session: clone; re-run: pull the latest code.
%cd /kaggle/working
!if [ -d pa2 ]; then git -C pa2 pull; else git clone -b pa2 https://github.com/rana-taqveem/atml_internal.git pa2; fi
%cd /kaggle/working/pa2
!pip install -q -r requirements.txt
!python -m scripts.download_assets
!python -m scripts.validate_assets

In [ ]:
!python -m scripts.check_environment   # expect CUDA available: True, GPU: Tesla T4

## 2. Smoke test (~3 min)

In [ ]:
!python -m task1_dpo.train --config configs/dpo.yaml --run-name smoke --output outputs/task1_dpo/smoke --max-examples 64
!python -m task1_dpo.evaluate --config configs/dpo.yaml --adapter outputs/task1_dpo/smoke --name smoke --n-gen-prompts 16
!head -30 results/task1_dpo/smoke/train_summary.json

In [ ]:
!rm -rf outputs/task1_dpo/smoke results/task1_dpo/smoke

## 3. Step 1 - standard DPO (1 epoch) + SFT baseline

In [ ]:
!python -m task1_dpo.train --config configs/dpo.yaml --run-name standard 2>&1 | tee -a task1.log
!python -m task1_dpo.evaluate --config configs/dpo.yaml --adapter none --name sft 2>&1 | tee -a task1.log
!python -m task1_dpo.evaluate --config configs/dpo.yaml --adapter outputs/task1_dpo/standard --name standard 2>&1 | tee -a task1.log

## 4. Step 2 - beta forks (600 pairs each)

In [ ]:
!python -m task1_dpo.ablate_beta --config configs/dpo.yaml --skip-existing 2>&1 | tee -a task1.log

## 5. Step 3 - length-balanced DPO + per-stratum / word-limit analysis

In [ ]:
!python -m task1_dpo.analyze_length --config configs/dpo.yaml --skip-existing 2>&1 | tee -a task1.log

## 6. Summary, figures, qualitative candidates

In [ ]:
!python -m task1_dpo.summarize --config configs/dpo.yaml 2>&1 | tee -a task1.log

## 7. Save (download `task1_outputs.zip` from the Output panel)

In [ ]:
!zip -qr /kaggle/working/task1_outputs.zip outputs/task1_dpo results/task1_dpo task1.log
!ls -lh /kaggle/working/task1_outputs.zip